In [58]:
from gurobipy import *
import os
import pandas as pd
import numpy as np

q = 10.0    # deployment time per hydrant connection (time units)
r = 0.058   # illustrative flow decay per 100 m of hose

os.environ["GRB_LICENSE_FILE"] = "/Users/a_voorhaar/desktop/gurobi.lic"


In [59]:
df = pd.read_csv("candidates.csv")
df['Hydrant'] = df['Hydrant'].str.replace('H', '').astype(int)
df = df.set_index('Hydrant')
df


,Distance_m,Capacity_L_min
Hydrant,,
479,0.000000,2700
484,94.622568,2100
596,112.676814,900
476,115.788840,1400
592,143.730618,1200
491,148.311431,1000
475,197.887328,1800


In [60]:
def available(original_df, unavailable_indices):
    """Return a copy of `original_df` with `unavailable_indices` removed."""
    new_df = original_df.copy()
    if unavailable_indices:
        valid = [idx for idx in unavailable_indices if idx in new_df.index]
        if valid:
            new_df = new_df.drop(valid)
    return new_df


In [61]:
def new_search(current_config, dataframe):
    """Compute remaining capacity, surviving used hydrants, and reduced dataframe.

    Returns:
        (remaining_capacity, remaining_df, surviving)
    """
    used = list(current_config.get("used", []))
    unavailable = list(current_config.get("unavailable", []))

    # Hydrants that are both used and unavailable drop out of "used".
    surviving = [h for h in used if h not in unavailable]

    # Deduct the surviving hydrants' capacity from the required capacity.
    total_capacity = sum(
        np.exp(-r * dataframe.loc[h, "Distance_m"] / 100.0) * dataframe.loc[h, "Capacity_L_min"]
        for h in surviving
        if h in dataframe.index
    )
    remaining_capacity = current_config["capacity_needed"] - total_capacity

    # Candidates = dataframe minus the union of used and unavailable.
    exclude = sorted(set(surviving) | set(unavailable))
    remaining_df = available(dataframe, exclude)

    return remaining_capacity, remaining_df, surviving


In [62]:
def get_hydrant_recommendations(R, data, v, verbose=True):
    """Run Gurobi to pick hydrants covering at least `R` capacity.

    Minimizes sum(distance/v + q) subject to decayed delivered capacity.
    Returns (objective_value, selected_hydrants) or (None, None).
    """
    if v == 0:
        raise ValueError("v must be non-zero")

    m = Model("hydrant_selection")
    x = m.addVars(data.index, vtype=GRB.BINARY, name="x")

    m.setObjective(quicksum((data.loc[h, 'Distance_m'] / v + q) * x[h] for h in data.index), GRB.MINIMIZE)
    m.addConstr(quicksum(np.exp(-r * data.loc[h, 'Distance_m'] / 100.0) * data.loc[h, 'Capacity_L_min'] * x[h] for h in data.index) >= R, "capacity_constraint")

    m.Params.OutputFlag = 0
    m.optimize()

    if m.status == GRB.OPTIMAL:
        selected_hydrants = [h for h in data.index if x[h].X > 0.5]
        total_capacity = sum(data.loc[h, 'Capacity_L_min'] for h in selected_hydrants)
        if verbose:
            print(f"\nOptimal objective value for R={R}: {m.objVal:.2f} time units")
            print("Selected hydrants:")
            for h in selected_hydrants:
                print(f"  Hydrant {h}: Objective value = {data.loc[h, 'Distance_m']} time units, "
                      f"Capacity = {data.loc[h, 'Capacity_L_min']} L/min")
            print(f"Total capacity of selected hydrants: {total_capacity} L/min (Required: {R} L/min)")
        return m.objVal, selected_hydrants

    if verbose:
        if m.status == GRB.INFEASIBLE:
            print(f"Model is infeasible for R={R}, no solution found.")
        elif m.status == GRB.UNBOUNDED:
            print(f"Model is unbounded for R={R}.")
        else:
            print(f"Optimization ended with status {m.status} for R={R}")
    return None, None


In [63]:
class HydrantSearch:
    """Reusable recommender that re-solves as hydrant availability changes.

    A configuration holds:
        used            -> hydrants already committed to the solution
        unavailable     -> hydrants known to be out of service
        capacity_needed -> fixed total capacity the solution must cover
    """

    def __init__(self, dataframe, capacity_needed, v, used=None, unavailable=None):
        self.dataframe = dataframe
        self.v = v
        self.config = {
            "used": list(used) if used is not None else [],
            "unavailable": list(unavailable) if unavailable is not None else [],
            "capacity_needed": capacity_needed,
        }
        self.history = []

    def _capacity_of(self, hydrants):
        """Decayed (effective) flow of the given hydrants."""
        return sum(
            np.exp(-r * self.dataframe.loc[h, "Distance_m"] / 100.0) * self.dataframe.loc[h, "Capacity_L_min"]
            for h in hydrants
            if h in self.dataframe.index
        )

    def reanalyze(self, unavailable=None, capacity_needed=None, accept=True):
        if capacity_needed is not None:
            self.config["capacity_needed"] = capacity_needed
        if unavailable:
            for h in unavailable:
                if h not in self.config["unavailable"]:
                    self.config["unavailable"].append(h)

        remaining_capacity, remaining_df, surviving_used = new_search(self.config, self.dataframe)

        new_selected = []
        objective = None
        infeasible = False
        if remaining_capacity > 0:
            objective, new_selected = get_hydrant_recommendations(
                remaining_capacity, remaining_df, self.v, verbose=False
            )
            if new_selected is None:
                new_selected = []
                infeasible = True

        full_solution = sorted(set(surviving_used) | set(new_selected))

        if accept and not infeasible:
            self.config["used"] = full_solution

        result = {
            "surviving_used": sorted(surviving_used),
            "new_selected": sorted(new_selected),
            "full_solution": full_solution,
            "remaining_capacity": remaining_capacity,
            "total_capacity": self._capacity_of(full_solution),
            "objective": objective,
            "infeasible": infeasible,
        }
        self.history.append(result)
        self._print_summary(result)
        return result

    def _print_summary(self, result):
        print("Recommendation:")
        print(f"  Still-working used hydrants: {result['surviving_used']} "
              f"({self._capacity_of(result['surviving_used']):.1f} L/min effective)")
        print(f"  Newly selected: {result['new_selected']}")
        print(f"  Full solution: {result['full_solution']}")
        print(f"  Effective flow: {result['total_capacity']:.1f} L/min "
              f"(needed: {self.config['capacity_needed']} L/min)")
        if result["infeasible"]:
            print("  Warning: remaining hydrants cannot cover the required capacity.")
        elif result["objective"] is not None:
            print(f"  Objective value: {result['objective']:.2f} time units")


In [64]:
hs = HydrantSearch(df, capacity_needed=4000, v=5)
hs.reanalyze()
hs.reanalyze(unavailable=[479])
hs.reanalyze(capacity_needed=5000)


Recommendation:
  Still-working used hydrants: [] (0.0 L/min effective)
  Newly selected: [479, 484]
  Full solution: [479, 484]
  Effective flow: 4687.9 L/min (needed: 4000 L/min)
  Objective value: 38.92 time units
Recommendation:
  Still-working used hydrants: [484] (1987.9 L/min effective)
  Newly selected: [476, 596]
  Full solution: [476, 484, 596]
  Effective flow: 4140.0 L/min (needed: 4000 L/min)
  Objective value: 65.69 time units
Recommendation:
  Still-working used hydrants: [476, 484, 596] (4140.0 L/min effective)
  Newly selected: [592]
  Full solution: [476, 484, 592, 596]
  Effective flow: 5244.0 L/min (needed: 5000 L/min)
  Objective value: 38.75 time units


{'surviving_used': [476, 484, 596],
 'new_selected': [592],
 'full_solution': [476, 484, 592, 596],
 'remaining_capacity': np.float64(860.0143335214452),
 'total_capacity': np.float64(5244.00537339041),
 'objective': 38.74612366714889,
 'infeasible': False}